# Local VLM evaluation — image download pack

This notebook **does not run a VLM**. It prepares a downloadable ZIP of generated images with filenames that clearly encode generation and LoRA condition. Download the ZIP from the notebook output, then evaluate it on your own PC.

Naming rule: `gNN_genNNN_lora.png` or `gNN_genNNN_nolora.png`.
- `g00`: Shishi single-image group
- `g01`: girl / subject comparison group
- `genNNN`: generation number, zero-padded
- `lora` / `nolora`: LoRA enabled / disabled

The original PNGs are preserved.

In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

WORK = Path('/kaggle/working')
PACK_DIR = WORK / 'local_vlm_pack'
IMAGES_DIR = PACK_DIR / 'images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Known files produced by comfy-gen-lora. Extend this mapping for future generations.
# g00 = Shishi; g01 = girl comparison. Condition is explicit in every output filename.
sources = [
    {
        'source': WORK / 'yamazaki_shishi_lora_test.png',
        'group': 'g00',
        'generation': 1,
        'condition': 'lora',
        'subject': 'shishi',
    },
    {
        'source': WORK / 'yamazaki_shishi_lora_vlm_test.png',
        'group': 'g01',
        'generation': 1,
        'condition': 'lora',
        'subject': 'girl',
    },
    {
        'source': WORK / 'yamazaki_shishi_baseline_no_lora.png',
        'group': 'g01',
        'generation': 1,
        'condition': 'nolora',
        'subject': 'girl',
    },
]

manifest = []
missing = []
for item in sources:
    src = item['source']
    if not src.is_file():
        missing.append(str(src))
        continue
    filename = f"{item['group']}_gen{item['generation']:03d}_{item['condition']}.png"
    dest = IMAGES_DIR / filename
    shutil.copy2(src, dest)
    manifest.append({
        'group': item['group'],
        'generation': item['generation'],
        'subject': item['subject'],
        'condition': item['condition'],
        'filename': f'images/{filename}',
        'source_filename': src.name,
    })

if not manifest:
    raise FileNotFoundError(
        'No known generated PNGs found in /kaggle/working. Run the generation notebook first.'
    )

(PACK_DIR / 'manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + '\\n',
    encoding='utf-8'
)
(PACK_DIR / 'README.txt').write_text(
    'Local VLM evaluation image pack\\n'
    'Naming: gNN_genNNN_lora.png or gNN_genNNN_nolora.png\\n'
    'g00=Shishi, g01=girl comparison; genNNN=generation index; lora/nolora=LoRA condition.\\n'
    'See manifest.json for source filenames.\\n',
    encoding='utf-8'
)

zip_path = WORK / 'local_vlm_pack.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(PACK_DIR.rglob('*')):
        if path.is_file():
            zf.write(path, path.relative_to(WORK))

print('Download this file from Kaggle Output:', zip_path)
print('Included images:')
for row in manifest:
    print('-', row['filename'], '<-', row['source_filename'])
if missing:
    print('Skipped missing source files:')
    for path in missing:
        print('-', path)


## Naming convention\n\n| Example | Meaning |\n|---|---|\n| `g00_gen001_lora.png` | Shishi, generation 1, LoRA enabled |\n| `g01_gen001_lora.png` | Girl comparison, generation 1, LoRA enabled |\n| `g01_gen001_nolora.png` | Girl comparison, generation 1, LoRA disabled |\n\nThis is a filename convention, not a claim that images have been visually verified. To add another generation, increment `generation` in the source mapping and keep the same prompt/seed/settings for matched LoRA/no-LoRA comparisons.